[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/ai-ml-interviews-study-guide/blob/main/notebooks/week5_eval_overfitting.ipynb)


# Week 5 — Why Eval Leaderboards Are Overfitted

**Thesis.** Evaluating *K* candidate models on a single held-out test set is the ML analogue of running *K* backtests on the same price history. The observed best score is not an unbiased estimate of the best model's true skill — it is upwardly biased by a selection effect whose magnitude scales like $\sqrt{2\log K}$ for Gaussian-ish noise. This is the same trick Lopez de Prado calls the **Deflated Sharpe Ratio** in *Advances in Financial Machine Learning* (Ch. 11 / Ch. 14): if you sift through enough strategies, the best one *will* look great even if none of them have any edge.

In this notebook we:
1. Simulate K=50 zero-skill models on one synthetic test set.
2. Show that the apparent SOTA accuracy is well above the true accuracy.
3. Derive the analytic deflation $\sqrt{p(1-p)/n}\cdot\sqrt{2\log K}$.
4. Confirm it Monte-Carlo style.
5. Define a `deflated_accuracy` and watch the leaderboard collapse.
6. Sweep K to show how the bias grows.
7. Note where the iid bound under-estimates real-world overfitting.

**Interview payoff:** "What's wrong with current eval methodology?" — answered at the bottom in 90 seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(99)

## 1. Setup — K zero-skill candidates

K = 50 models. Each has true accuracy $p_{\text{true}} = 0.70$ on a benchmark of $n=500$ items. We score each model on the *same* test set, but for clean intuition we simulate each model's per-item correctness independently. (This *exaggerates* the selection effect compared to fully-correlated models; the directional point still holds, and we discuss the gap at the end.)

In [ ]:
K = 50
n = 500
p_true = 0.70

# (K, n) matrix of Bernoulli(p_true) outcomes
correct = np.random.binomial(1, p_true, size=(K, n))
obs_acc = correct.mean(axis=1)

print(f"K = {K} models, n = {n} items, true accuracy p_true = {p_true}")
print(f"Mean observed accuracy across models: {obs_acc.mean():.4f}")
print(f"Max observed accuracy (apparent SOTA): {obs_acc.max():.4f}")
print(f"Min observed accuracy:                 {obs_acc.min():.4f}")

## 2. Naive leaderboard

Sort by observed accuracy, print the top 5. Notice the gap between rank 1 and the true skill 0.70.

In [ ]:
order = np.argsort(-obs_acc)
print("Rank | model_id | observed_acc | gap_vs_truth")
print("-" * 50)
for rank, idx in enumerate(order[:5], start=1):
    print(f"{rank:>4} | {idx:>8} | {obs_acc[idx]:.4f}       | +{obs_acc[idx] - p_true:+.4f}")

apparent_sota = obs_acc.max()
print(f"\nApparent SOTA = {apparent_sota:.4f}")
print(f"True skill   = {p_true:.4f}")
print(f"Selection-induced inflation = {apparent_sota - p_true:+.4f}")

## 3. Expected max over K candidates — theory

For K i.i.d. standard normal draws $Z_1, \ldots, Z_K$,
$$\mathbb{E}[\max_i Z_i] \approx \sqrt{2 \log K} \quad \text{(leading order, large } K\text{)}.$$

Each model's observed accuracy is approximately $\mathcal{N}\big(p_{\text{true}},\, \sigma^2\big)$ with $\sigma = \sqrt{p_{\text{true}}(1-p_{\text{true}})/n}$ by the CLT. Therefore
$$\mathbb{E}[\max_i \hat p_i] \approx p_{\text{true}} + \sigma \sqrt{2 \log K} = p_{\text{true}} + \sqrt{\tfrac{p_{\text{true}}(1-p_{\text{true}})}{n}}\sqrt{2 \log K}.$$

This is exactly the structural form of Lopez de Prado's **Deflated Sharpe Ratio**: subtract a $\sqrt{2 \log K}$-style term from the best observed metric to undo the look-elsewhere effect.

In [ ]:
sigma = np.sqrt(p_true * (1 - p_true) / n)
analytic_bias = sigma * np.sqrt(2 * np.log(K))

print(f"Per-model std sigma           = {sigma:.5f}")
print(f"sqrt(2 log K) for K={K}        = {np.sqrt(2*np.log(K)):.4f}")
print(f"Analytic selection bias       = {analytic_bias:.5f}")
print(f"=> expected apparent SOTA     ~ {p_true + analytic_bias:.4f}")
print(f"   (vs observed apparent SOTA = {apparent_sota:.4f})")

## 4. Empirical confirmation — Monte Carlo

Repeat the experiment 2,000 times. Record max observed accuracy each time. Plot the distribution against the analytic bound.

In [ ]:
n_trials = 2000
max_accs = np.empty(n_trials)
for t in range(n_trials):
    sims = np.random.binomial(1, p_true, size=(K, n)).mean(axis=1)
    max_accs[t] = sims.max()

empirical_mean_max = max_accs.mean()
print(f"Empirical E[max acc] over {n_trials} trials = {empirical_mean_max:.4f}")
print(f"Analytic prediction                      = {p_true + analytic_bias:.4f}")
print(f"Truth                                    = {p_true:.4f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(max_accs, bins=40, color="#88a", edgecolor="white", alpha=0.85)
ax.axvline(p_true, color="black", linestyle="--", label=f"p_true = {p_true:.3f}")
ax.axvline(p_true + analytic_bias, color="crimson", linestyle="-",
           label=f"p_true + sqrt(p(1-p)/n)*sqrt(2 log K) = {p_true+analytic_bias:.3f}")
ax.axvline(empirical_mean_max, color="forestgreen", linestyle=":",
           label=f"empirical mean of max = {empirical_mean_max:.3f}")
ax.set_xlabel("max observed accuracy across K=50 zero-skill models")
ax.set_ylabel("frequency")
ax.set_title("Distribution of apparent SOTA when no model has any real edge")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

The analytic line lands right next to the empirical mean. The leftmost dashed line (the truth) is well below where the leaderboard winner ends up. **Every single model in this experiment has zero edge over every other model**, yet the winner looks ~2 standard errors better than truth.

## 5. A deflated metric

Following the spirit of the Deflated Sharpe Ratio, define:
$$\text{deflated\_accuracy}(\hat p_{\max}; K, n, \hat p) = \hat p_{\max} - \sqrt{\tfrac{\hat p(1-\hat p)}{n}}\sqrt{2\log K}.$$

In [ ]:
def deflated_accuracy(observed_max, K, n, p_hat):
    sigma_hat = np.sqrt(p_hat * (1 - p_hat) / n)
    return observed_max - sigma_hat * np.sqrt(2 * np.log(K))

# Use the mean observed accuracy as p_hat (the pooled best-guess of skill).
p_hat = obs_acc.mean()
deflated_sota = deflated_accuracy(apparent_sota, K, n, p_hat)

print(f"Apparent SOTA (rank-1 raw):      {apparent_sota:.4f}")
print(f"Deflation term:                  {apparent_sota - deflated_sota:.4f}")
print(f"Deflated SOTA:                   {deflated_sota:.4f}")
print(f"True skill (ground truth):       {p_true:.4f}")
print()
print("Top-5 leaderboard, raw vs deflated:")
print("Rank | obs_acc | deflated")
for rank, idx in enumerate(order[:5], start=1):
    d = deflated_accuracy(obs_acc[idx], K, n, p_hat)
    print(f"{rank:>4} | {obs_acc[idx]:.4f}  | {d:.4f}")

After deflation the SOTA collapses back toward $p_{\text{true}} = 0.70$. The leaderboard's apparent edge was almost entirely selection bias.

## 6. How the bias scales with K

Vary K from 1 to 500. For each K, average the max observed accuracy across many MC trials, and overlay the analytic curve $p_{\text{true}} + \sigma\sqrt{2\log K}$. The more models you compare, the more you have to deflate.

In [ ]:
Ks = np.array([1, 2, 5, 10, 20, 50, 100, 500])
trials_per_K = 800
emp_means = np.empty(len(Ks))
for i, k in enumerate(Ks):
    sims = np.random.binomial(1, p_true, size=(trials_per_K, k, n)).mean(axis=2)
    emp_means[i] = sims.max(axis=1).mean()

analytic = p_true + sigma * np.sqrt(2 * np.log(np.maximum(Ks, 1.0001)))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(Ks, emp_means, "o-", color="forestgreen", label="empirical E[max obs acc]")
ax.plot(Ks, analytic, "s--", color="crimson",
        label=r"analytic $p + \sigma\sqrt{2\log K}$")
ax.axhline(p_true, color="black", linestyle=":", label=f"true skill {p_true}")
ax.set_xscale("log")
ax.set_xlabel("K (number of candidate models)")
ax.set_ylabel("max observed accuracy")
ax.set_title("Apparent SOTA grows with K even when nobody has any edge")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

for k, e, a in zip(Ks, emp_means, analytic):
    print(f"K={k:>4d}  empirical={e:.4f}  analytic={a:.4f}")

## 7. When the iid assumption breaks

The bound above is *optimistic* about how much you can trust a leaderboard, for three reasons:

1. **Candidates are not independent.** Real model variants share architecture, pre-training data, and tokenizer. Their errors are correlated, so the *effective* number of independent trials $K_{\text{eff}}$ is smaller than the headline K — but the *variance reduction* from sharing data is also smaller. In practice, for closely-related siblings the effective K can be dramatically lower; for highly diverse zoos it can be close to K.
2. **The test set has finite capacity.** Once a benchmark has been hill-climbed for years, public test sets leak into training corpora (directly or via blog posts, papers, eval harness leaks). True $n$ shrinks; effective bias grows.
3. **Selection happens iteratively.** Real research isn't "propose K models, evaluate once." It's "propose, evaluate, ablate, tweak, evaluate, repeat." Every glance at the test set during iteration is another sample in the multiple-comparisons pool. The effective K is the number of *decisions* you made looking at the test set, which is enormous.

All three push the real deflation **larger** than the simple $\sqrt{2\log K}$ bound here. This is exactly the **backtest overfitting** story Lopez de Prado tells about quant strategies, ported one-for-one to ML eval.

## 8. Flashcard / 90-second pitch

> **Deflated Sharpe Ratio for ML eval.** If you score K candidate models on the same test set, the maximum observed metric is biased upward by roughly $\sigma \sqrt{2\log K}$, where $\sigma$ is the per-model sampling standard error. In our toy run with K=50 zero-skill models, true accuracy 0.70, n=500: the apparent SOTA was about **0.74**, the analytic deflation was about **0.028**, and after subtracting it the deflated SOTA was about **0.71** — within sampling noise of the truth. None of the 50 models actually had any edge. This is identical in spirit to Lopez de Prado's Deflated Sharpe Ratio for backtest selection: hill-climbing on a fixed eval is the same statistical sin as hill-climbing on a fixed price history. The fix is the same too: subtract the look-elsewhere term, and report intervals, not point estimates.

## 9. Interview rehearsal — "What's wrong with current eval methodology?"

> Three things, in order of severity. **One**, the multiple-comparisons problem. Every benchmark is evaluated by hundreds of models; the leader is by construction an outlier. The bias scales like $\sigma\sqrt{2\log K}$, which is exactly the structure Lopez de Prado uses for the Deflated Sharpe Ratio in quant finance. In a toy run I did this week — 50 models, 500-item test, all with identical 0.70 true accuracy — the apparent SOTA came in around 0.74 and the analytic deflation was about 0.028; after deflating, the leaderboard collapsed back to ~0.71, which is just noise around truth. **Two**, the iid assumption is generous. Real models are correlated, real test sets leak into training, and real research iterates on the test set hundreds of times — the effective K is way bigger than the K you see, so the real deflation is worse than my bound. **Three**, almost no public leaderboard reports a confidence interval, let alone a deflated metric. The fix is mechanical: report $\hat p \pm 1.96\sigma$, deflate the rank-1 score by $\sigma\sqrt{2\log K}$ where K is the number of models that have ever touched the test set, and treat any improvement smaller than the deflation as zero. Until the field does that, most reported SOTA gains are the ML version of an over-fit backtest.